# Classical and Modern Applied Cryptography
**Student:** Mahfuzur Rahman | **ID:** IT-23005  
**Course:** Cryptography and Cyber Law  
**Reference:** [IIT Bombay MNP Club - Number Theory & Cryptography Workshop](https://github.com/mnp-club/Number_Theory_and_Cryptography_Workshop)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IT23005/Cryptography-and-Cyber-Law/blob/main/CT-1%20Practicing%20Number%20Theory/02_Classical_and_Modern_Cryptography.ipynb)

---
## Overview
This notebook covers classical information-theoretic security and public-key cryptosystems:
1. **One-Time Pad (OTP) & Shannon's Perfect Secrecy**
2. **The Many-Time Pad Vulnerability & Crib Dragging Attack**
3. **Diffie-Hellman Key Exchange Protocol**
4. **RSA Cryptosystem: Key Generation, Encryption, Decryption**
5. **Shor's Quantum Factoring Algorithm (Order-Finding Reduction)**


### 1. One-Time Pad & String Conversions
Text to hex conversion, bitwise XOR, and perfect encryption.


In [ ]:
from functools import reduce

def toHex(s):
    return ''.join(f"{ord(c):02x}" for c in s)

def toStr(hex_str):
    if len(hex_str) % 2 != 0:
        hex_str = '0' + hex_str
    return bytes.fromhex(hex_str).decode('latin1')

def Xor(hex1, hex2):
    assert len(hex1) == len(hex2)
    b1 = bytes.fromhex(hex1)
    b2 = bytes.fromhex(hex2)
    return bytes(x ^ y for x, y in zip(b1, b2)).hex()

message = "secret message"
key     = "my secret keys"
c = Xor(toHex(message), toHex(key))
print("Ciphertext hex:", c)
decrypted = toStr(Xor(c, toHex(key)))
print("Decrypted msg :", decrypted)
assert decrypted == message


### 2. Many-Time Pad (Two-Time Pad Attack)
When key $K$ is re-used to encrypt two messages $M_1$ and $M_2$:
$$C_1 = M_1 \oplus K, \quad C_2 = M_2 \oplus K \implies C_1 \oplus C_2 = M_1 \oplus M_2$$
The key cancels out! An eavesdropper can use **crib dragging** to slide guessed English words across the XOR stream.


In [ ]:
message1 = "steal the secret"
message2 = "the cat the bird"
key      = "supersecretverys"

ciphertext1 = Xor(toHex(message1), toHex(key))
ciphertext2 = Xor(toHex(message2), toHex(key))

xor_ciphertexts = Xor(ciphertext1, ciphertext2)
xor_messages    = Xor(toHex(message1), toHex(message2))
assert xor_ciphertexts == xor_messages
print("XOR of ciphertexts equals XOR of messages:", xor_ciphertexts == xor_messages)

def TryGuessingSubstring(substring, message_length, xor_messages_hex):
    good_guesses = []
    sub_len = len(substring)
    for pos in range(message_length - sub_len + 1):
        padded_guess = chr(0) * pos + substring + chr(0) * (message_length - sub_len - pos)
        guess_hex = toHex(padded_guess)
        other_part = toStr(Xor(guess_hex, xor_messages_hex))[pos:pos + sub_len]
        if all(ch.isalpha() or ch.isspace() for ch in other_part):
            good_guesses.append((pos, substring, other_part))
    return good_guesses

guesses = TryGuessingSubstring(" the ", len(message1), xor_ciphertexts)
print("
Crib Dragging Results for ' the ':")
for pos, crib, other in guesses:
    print(f"  Position {pos}: One message has '{crib}', other message has '{other}'")


### 3. Diffie-Hellman Key Exchange Protocol
Simulation of Alice and Bob establishing a shared secret over an insecure channel.


In [ ]:
N = 11    # Public Prime Modulus
g = 7     # Public Generator (Primitive Root modulo 11)

# Alice and Bob pick private keys
a = 2     # Alice's private key
b = 4     # Bob's private key

# Public values transmitted over network
A = pow(g, a, N)    # Alice transmits A
B = pow(g, b, N)    # Bob transmits B

print("Alice transmits A =", A)
print("Bob transmits B   =", B)

# Each derives the shared secret
K_alice = pow(B, a, N)
K_bob   = pow(A, b, N)

assert K_alice == K_bob
print("Shared Synthesized Key =", K_alice)
print("Why it works: (g^b)^a = (g^a)^b = g^(ab) mod N")


### 4. RSA Cryptosystem Implementation
Full pipeline: Text encoding to large integer, Key generation, Encryption, Decryption, and Totient Verification.


In [ ]:
def ConvertToInt(s):
    res = 0
    for ch in s: res = res * 256 + ord(ch)
    return res

def ConvertToStr(n):
    res = []
    while n > 0:
        res.append(chr(n % 256))
        n //= 256
    return "".join(reversed(res))

def ExtendedEuclid(a, b):
    if b == 0: return (1, 0)
    (x, y) = ExtendedEuclid(b, a % b)
    k = a // b
    return (y, x - k * y)

def InvertModulo(a, n):
    (b, x) = ExtendedEuclid(a, n)
    if b < 0: b = (b % n + n) % n
    return b

p = 1000000000039
q = 1000000000061
N = p * q
phiN = (p - 1) * (q - 1)
e = 65537
d = InvertModulo(e, phiN)

print("Modulus N :", N)
print("Public Key (e, N) :", (e, N))
print("Private Key (d, N):", (d, N))

msg = "attack"
m_int = ConvertToInt(msg)
c_int = pow(m_int, e, N)
decrypted_int = pow(c_int, d, N)
decrypted_msg = ConvertToStr(decrypted_int)

print("
Plaintext :", msg)
print("Ciphertext:", c_int)
print("Decrypted :", decrypted_msg)
assert decrypted_msg == msg


### 5. Shor's Quantum Factoring Algorithm Simulation
Classical reduction: How quantum period-finding factors $N = p \cdot q$ using $\gcd(a^{r/2} \pm 1, N)$.


In [ ]:
import math

N = 35
a = 8  # gcd(8, 35) = 1

# Finding period r such that a^r = 1 mod N
# On a quantum computer, this is performed in O((log N)^2) using Quantum Fourier Transform
r = 1
while pow(a, r, N) != 1:
    r += 1

print(f"Period r of {a} modulo {N} is: {r}")
assert r % 2 == 0, "Period must be even"
half_val = pow(a, r // 2, N)
assert half_val != N - 1, "Trivial factor condition"

# Factors extracted via GCD:
factor1 = math.gcd(half_val - 1, N)
factor2 = math.gcd(half_val + 1, N)

print(f"Extracted Factor 1: gcd({half_val} - 1, {N}) = {factor1}")
print(f"Extracted Factor 2: gcd({half_val} + 1, {N}) = {factor2}")
assert factor1 * factor2 == N
print(f"Successfully factored {N} into {factor1} x {factor2}!")
